## 07 — Communicate Results
Confusion Matrix, ROC Curve, Feature Importance và danh sách đại lý rủi ro cao.

In [ ]:
# Confusion Matrix + ROC Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Active", "Churn"], ax=ax1)
ax1.set_title("Confusion Matrix — XGBoost")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax2, name="XGBoost")
ax2.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax2.set_title("ROC Curve — XGBoost")
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Feature Importance
fi = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
fi.plot(kind="barh", ax=ax, alpha=0.85)
ax.set_xlabel("Importance (Gain)")
ax.set_title("Feature Importance — XGBoost Churn Model")
ax.grid(axis="x", alpha=0.4)
fig.tight_layout()
plt.show()

print(f"Đặc trưng quan trọng nhất: {fi.idxmax()} ({fi.max():.3f})")

In [ ]:
# Danh sách đại lý rủi ro
rfm["churn_prob"] = clf.predict_proba(X_scaled)[:, 1]
rfm["risk_level"] = pd.cut(rfm["churn_prob"], bins=[0,.3,.6,1.], labels=["Thấp","Trung bình","Cao"])

name_map = df.drop_duplicates("customer_code")[["customer_code","customer_name"]]
rfm_out  = rfm.merge(name_map, on="customer_code", how="left")

risk_counts = rfm["risk_level"].value_counts().reindex(["Thấp","Trung bình","Cao"])
print("Phân loại rủi ro:")
for level, cnt in risk_counts.items():
    print(f"  {level:<12}: {cnt} đại lý ({cnt/len(rfm)*100:.1f}%)")

In [ ]:
# Biểu đồ phân loại rủi ro
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(rfm["churn_prob"], bins=25, alpha=0.8, edgecolor="white")
ax1.axvline(0.3, linestyle="--", linewidth=2, label="0.3")
ax1.axvline(0.6, linestyle=":",  linewidth=2, label="0.6")
ax1.set_xlabel("Xác suất Churn")
ax1.set_ylabel("Số đại lý")
ax1.set_title("Phân bố Xác suất Churn")
ax1.legend()
ax1.grid(axis="y", alpha=0.4)

ax2.bar(risk_counts.index, risk_counts.values, alpha=0.85)
for i, v in enumerate(risk_counts.values):
    ax2.text(i, v + 1, str(v), ha="center", fontsize=11, fontweight="bold")
ax2.set_xlabel("Mức độ rủi ro")
ax2.set_ylabel("Số đại lý")
ax2.set_title("Phân loại Rủi ro Churn")
ax2.grid(axis="y", alpha=0.4)

fig.tight_layout()
plt.show()

In [ ]:
# Lưu kết quả
import os
os.makedirs("output", exist_ok=True)

rfm_out[["customer_code","customer_name","recency","frequency","monetary",
         "last3m_orders","churn_prob","churn","risk_level"]]    .sort_values("churn_prob", ascending=False)    .to_csv("output/predictions_churn.csv", index=False)

print("Đã lưu output/predictions_churn.csv")
print()
print("Top 5 đại lý rủi ro cao nhất:")
print(rfm_out.sort_values("churn_prob", ascending=False)
      [["customer_code","customer_name","recency","churn_prob"]]
      .head(5).to_string(index=False))